# Chạy liên tục 30 video trên Colab và ghi thời gian vào CSV/Excel

Notebook này chạy **đúng pipeline của worker** (`ai_workers.tasks.process_video`: ASR, cảnh và keyframe, OCR, Florence-2, YOLO11n phần tử slide, timeline, LED) lần lượt cho từng video, nhưng không cần website, Redis hay R2. Kết quả mỗi video được ghi ngay vào một file CSV trên Google Drive, nên mất kết nối giữa chừng không mất dữ liệu: chạy lại cell chạy là nó bỏ qua video đã xong.

**Cần chuẩn bị**
- Runtime → Change runtime type → GPU (T4 trở lên).
- Không có secret bắt buộc. Tùy chọn: `HF_TOKEN` (phân biệt người nói), `SLIDE_ELEMENT_DETECTOR_WEIGHTS`, `ENABLE_OBJECT_TRACKING`... giống notebook worker. Checkpoint YOLO11n phải có trên Drive như notebook worker.
- Danh sách mặc định gồm 30 video YouTube lấy từ báo cáo nghiên cứu (1 mốc đã chạy, 13 talk học thuật, 4 bài giảng, 3 đối chứng, 9 video có mốc chuyển slide). Sửa ô danh sách nếu bạn muốn video khác.

**Thời gian** (ước tính, chưa đo cho cả danh sách): các video này dài tổng cộng khoảng 11,8 giờ; job đã chạy trên T4 mất khoảng 0,4 lần thời lượng video, nên cả lượt cần chừng 4 đến 5 giờ, chưa kể tải video. Chạy thử trước bằng `LIMIT = 2`.

**Video tải sẵn được ưu tiên.** Nếu `MyDrive/mls-batch/videos/<video_id>.mp4` có sẵn thì notebook dùng file đó và không chạm YouTube. **YouTube có thể chặn Colab:** video không có file sẵn mà bị chặn được ghi là `download_failed` và notebook chạy tiếp. Cách khắc phục: tải video bằng máy của bạn (`yt-dlp -f "bv*[height<=360][ext=mp4]+ba[ext=m4a]/b[height<=360]" -o "%(id)s.%(ext)s" <link>`) rồi đặt `<video_id>.mp4` vào thư mục `MyDrive/mls-batch/videos`; lần chạy sau notebook tự dùng file đó. Để tải cả danh sách một lần: chạy `python -m ai_workers.batch_download_videos` trên máy bạn, upload `videos.zip` vào `MyDrive/mls-batch/`, notebook tự giải nén.

**Đọc kết quả:** cột `stage_*_sec` là khoảng thời gian từ log đầu tiên đến log cuối cùng của từng giai đoạn. Nhánh audio và nhánh visual chạy song song nên các cột này chồng lên nhau, đừng cộng lại: tổng thời gian xử lý là `processing_sec`, và `realtime_factor` = `processing_sec` / `video_sec`. `video_sec` là thời điểm kết thúc của câu nói cuối cùng.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Kết quả và (tùy chọn) video tải sẵn nằm trên Drive; cache model cũng đặt ở đây để session sau không tải lại (~3 GB).
from google.colab import drive
drive.mount('/content/drive')
CACHE_DIR = '/content/drive/MyDrive/mls-cache'
BATCH_DIR = '/content/drive/MyDrive/mls-batch'
import os
os.makedirs(BATCH_DIR + '/videos', exist_ok=True)

# Video tải sẵn trên máy (ai_workers/batch_download_videos.py) được nén thành videos.zip và đặt trong mls-batch.
import zipfile
zip_path = BATCH_DIR + '/videos.zip'
if os.path.isfile(zip_path):
    with zipfile.ZipFile(zip_path) as archive:
        new = [n for n in archive.namelist() if not os.path.exists(BATCH_DIR + '/videos/' + n)]
        archive.extractall(BATCH_DIR + '/videos', members=new)
    print('videos.zip:', len(new), 'file mới được giải nén vào', BATCH_DIR + '/videos')

In [ ]:
import os, subprocess
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from google.colab import userdata

REPO_URL = 'https://github.com/multimodal-lecture-summarizer/multimodal-lecture-summarizer.git'
BRANCH = 'main'
REPO_DIR = '/content/mls'
PYTHON_BIN = '/content/venv/bin/python'

try:
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    token = None
url = REPO_URL.replace('https://', f'https://x-access-token:{token}@') if token else REPO_URL


def sh(cmd, **kwargs):
    subprocess.run(cmd, check=True, **kwargs)


def install_git_lfs():
    sh('apt-get update -qq && apt-get install -y -qq git-lfs', shell=True)


def clone_repo():
    # LFS files are fetched in the next phase, once git-lfs is installed.
    env = {**os.environ, 'GIT_LFS_SKIP_SMUDGE': '1'}
    if not os.path.isdir(REPO_DIR):
        sh(['git', 'clone', '--depth', '1', '--branch', BRANCH, url, REPO_DIR], env=env)
    else:
        # The runtime outlived an earlier run: move the checkout to the latest commit of BRANCH.
        sh(['git', 'fetch', '--depth', '1', 'origin', BRANCH], cwd=REPO_DIR, env=env)
        sh(['git', 'reset', '--hard', 'FETCH_HEAD'], cwd=REPO_DIR, env=env)
    sh(['git', 'log', '-1', '--oneline'], cwd=REPO_DIR)  # shows which commit this worker runs


def create_venv():
    if not Path(PYTHON_BIN).is_file():
        sh(['uv', 'venv', '--python', '3.11', '/content/venv'])


def pull_lfs():
    # Florence-2 weights live in Git LFS; the worker refuses to start without them.
    sh('git lfs install && git lfs pull', shell=True, cwd=REPO_DIR)


def install_requirements():
    sh(['uv', 'pip', 'install', '--python', PYTHON_BIN,
        '-r', 'ai_workers/requirements.txt', '-r', 'backend/requirements.txt',
        'setuptools'], cwd=REPO_DIR)  # PaddleOCR imports pkg_resources; uv venvs ship without setuptools
    # trackers declares numpy>=2 but runs on the pinned numpy 1.26.4, so it is installed without its deps.
    sh(['uv', 'pip', 'install', '--python', PYTHON_BIN, '--no-deps',
        '-r', 'ai_workers/requirements-tracking.txt'], cwd=REPO_DIR)


def run_parallel(*steps):
    with ThreadPoolExecutor(max_workers=len(steps)) as pool:
        for future in [pool.submit(step) for step in steps]:
            future.result()  # re-raises the first failure


# apt, clone and venv do not depend on each other; LFS needs apt+clone, install needs clone+venv.
run_parallel(install_git_lfs, clone_repo, create_venv)
run_parallel(pull_lfs, install_requirements)

sh([PYTHON_BIN, '-m', 'celery', '--version'])
print('Cài đặt hoàn tất.')

In [ ]:
import os
from google.colab import userdata

def secret(name):
    try:
        return userdata.get(name) or ''
    except Exception:
        return ''

def setting(name, default):
    """Công tắc không nhạy cảm: dùng Colab Secret cùng tên nếu có, không thì dùng mặc định."""
    return secret(name).strip() or default

cache_dir = globals().get('CACHE_DIR', '/content/mls/cache')
site = '/content/venv/lib/python3.11/site-packages'

# YOLO11n fine-tune trên LVVO (bảng, biểu đồ, ảnh, hình minh họa): checkpoint seed 1 trên Drive.
# Không thấy file (chưa mount Drive, hoặc đặt secret = 'off') thì bỏ qua bước này.
SLIDE_WEIGHTS_DEFAULT = '/content/drive/MyDrive/DetTrack/lvvo-ft-261115-v1/yolo/yolo11n_ft_seed1/weights/selected.pt'
slide_weights = setting('SLIDE_ELEMENT_DETECTOR_WEIGHTS', SLIDE_WEIGHTS_DEFAULT)
if slide_weights.lower() == 'off' or not os.path.isfile(slide_weights):
    if slide_weights.lower() != 'off':
        print('Không thấy checkpoint phát hiện phần tử slide, bỏ qua bước này:', slide_weights)
    slide_weights = ''

worker_env = {
    **os.environ,
    # Pipeline chạy trong một tiến trình: không cần Redis, kết quả đi vào CSV. Không đặt R2 nên bước
    # tải keyframe lên R2 bị bỏ qua (khoảng 2 giây mỗi video trên website).
    'CELERY_BROKER_URL': 'memory://',
    'CELERY_RESULT_BACKEND': 'cache+memory://',
    'HF_TOKEN': secret('HF_TOKEN'),
    'CACHE_DIR': cache_dir,
    'HF_HOME': f'{cache_dir}/huggingface',
    'FLORENCE_DEVICE': 'cuda',
    'SEMANTIC_CLIP_DEVICE': 'cuda',
    'PADDLEOCR_USE_GPU': '0',
    'ENABLE_PARALLEL_BRANCHES': setting('ENABLE_PARALLEL_BRANCHES', 'true'),
    'ENABLE_OBJECT_TRACKING': setting('ENABLE_OBJECT_TRACKING', 'false'),
    'SLIDE_ELEMENT_DETECTOR_WEIGHTS': slide_weights,
    'YOLO_CONFIG_DIR': f'{cache_dir}/ultralytics',
    'DENOISE_DEVICE': setting('DENOISE_DEVICE', 'cuda'),
    'ENABLE_LLM_SUMMARIZER': setting('ENABLE_LLM_SUMMARIZER', 'false'),
    'ASR_ENGLISH_BACKEND': setting('ASR_ENGLISH_BACKEND', 'qwen3'),
    'QWEN_ASR_BATCH_SIZE': setting('QWEN_ASR_BATCH_SIZE', '4'),
    'KEYFRAME_CANDIDATES': setting('KEYFRAME_CANDIDATES', '3'),
    'CUBLAS_WORKSPACE_CONFIG': ':4096:8',
    'PROCESS_MIN_AVAILABLE_MEMORY_MB': '4096',
    'PROCESS_SOFT_MIN_AVAILABLE_MEMORY_MB': '3072',
    'LD_LIBRARY_PATH': f'{site}/nvidia/cudnn/lib:{site}/nvidia/cublas/lib:' + os.environ.get('LD_LIBRARY_PATH', ''),
    'PATH': '/content/venv/bin:' + os.environ['PATH'],
}
os.makedirs(cache_dir, exist_ok=True)
switches = ('ENABLE_PARALLEL_BRANCHES', 'ENABLE_OBJECT_TRACKING', 'SLIDE_ELEMENT_DETECTOR_WEIGHTS', 'DENOISE_DEVICE',
            'ENABLE_LLM_SUMMARIZER', 'ASR_ENGLISH_BACKEND', 'QWEN_ASR_BATCH_SIZE', 'KEYFRAME_CANDIDATES')
print('Cấu hình xong. Cache:', cache_dir)
print('Công tắc:', {k: worker_env[k] for k in switches})

In [ ]:
# Nhóm: baseline = đã có kết quả trước đó; A = talk học thuật cùng kênh; B = bài giảng; C = video giải thích
# (ngoài phạm vi, dùng đối chứng); D = bài giảng có mốc chuyển slide của MLPDataset.
# Mỗi phần tử là (nhóm, link YouTube) hoặc (nhóm, đường dẫn file trên Drive / link tải trực tiếp).
VIDEOS = [
    ('baseline', 'https://www.youtube.com/watch?v=FLWvcdfGPxs'),  # OSDI '23 - Delegated User-level Virtualization (đã chạy trước đó)
    ('A', 'https://www.youtube.com/watch?v=MfSp7WJuz-8'),  # OSDI '23 - Core slicing
    ('A', 'https://www.youtube.com/watch?v=QY-83DX-R2I'),  # OSDI '23 - TraChe
    ('A', 'https://www.youtube.com/watch?v=OFjlUqHbrMQ'),  # OSDI '23 - Cilantro
    ('A', 'https://www.youtube.com/watch?v=tnkmpsAHyMg'),  # OSDI '23 - ExoFlow
    ('A', 'https://www.youtube.com/watch?v=snKg5vIYjb4'),  # OSDI '23 - Off-path SmartNIC
    ('A', 'https://www.youtube.com/watch?v=oyumGtYN558'),  # OSDI '23 - ShRing (từng bị YouTube chặn bot)
    ('A', 'https://www.youtube.com/watch?v=X929vAkovX4'),  # NSDI '23 - LOCA
    ('A', 'https://www.youtube.com/watch?v=QOkDuS9enUA'),  # NSDI '23 - DChannel
    ('A', 'https://www.youtube.com/watch?v=u3YG5hRQjdA'),  # NSDI '23 - Hermit
    ('A', 'https://www.youtube.com/watch?v=DcDnrZY11Yw'),  # FAST '23 - Perseus
    ('A', 'https://www.youtube.com/watch?v=zGnn7IkIdiE'),  # FAST '23 - Fast Application Launch
    ('A', 'https://www.youtube.com/watch?v=RfAq1JbHbXI'),  # USENIX ATC '23 - EnvPipe
    ('A', 'https://www.youtube.com/watch?v=HVRRL9Ky6Eo'),  # USENIX Security '23 - Ethical Frameworks
    ('B', 'https://www.youtube.com/watch?v=tA42nHmmEKw'),  # Python for Data Science
    ('B', 'https://www.youtube.com/watch?v=fC7V8QsPBec'),  # Intro to Machine Learning (Week 1)
    ('B', 'https://www.youtube.com/watch?v=XjOnY-OLAPc'),  # Quicksort, Timsort, Powersort - PyCon US 2023
    ('B', 'https://www.youtube.com/watch?v=II4giR4vOOo'),  # MIT 6.S191 Introduction to Deep Learning (3376s)
    ('C', 'https://www.youtube.com/watch?v=CqOfi41LfDw'),  # Essential Main Ideas of Neural Networks (ngoài phạm vi, đối chứng)
    ('C', 'https://www.youtube.com/watch?v=yIYKR4sgzI8'),  # StatQuest: Logistic Regression (ngoài phạm vi, đối chứng)
    ('C', 'https://www.youtube.com/watch?v=p-ltr1C7u2o'),  # ML Specialization Course 1 Week 3 (ngoài phạm vi, đối chứng)
    ('D', 'https://www.youtube.com/watch?v=IotJiLgIwvQ'),  # Anatomy & Physiology Chapter 10 Part D
    ('D', 'https://www.youtube.com/watch?v=2U3DsK7BHVU'),  # Anatomy & Physiology Chapter 9 Part B
    ('D', 'https://www.youtube.com/watch?v=RH-Pe9MBDBU'),  # Anatomy & Physiology Chapter 8 Part A
    ('D', 'https://www.youtube.com/watch?v=uM1JOFaATP4'),  # Chapter 32 Tissues and Endocrine System (3397s)
    ('D', 'https://www.youtube.com/watch?v=g1JIbOMWBBM'),  # Chapter 27 Rise of the Invertebrate Animals (3431s)
    ('D', 'https://www.youtube.com/watch?v=pUH2_SGnm9g'),  # Chapter 39 Musculoskeletal System
    ('D', 'https://www.youtube.com/watch?v=hPmpU72Cgsg'),  # Head & Neck Anatomy, Pharynx & Larynx Muscles
    ('D', 'https://www.youtube.com/watch?v=xcyUm5ZEzng'),  # Head & Neck Anatomy, Craniofacial Veins
    ('D', 'https://www.youtube.com/watch?v=Yeq6LO3q4Vo'),  # Periodontics, New Classification System
]

RUN_NAME = 'batch-30'        # tên file kết quả; đổi tên để chạy một lượt mới từ đầu
LIMIT = 0                    # đặt 2 để chạy thử 2 video đầu trước khi chạy cả danh sách
MAX_VIDEO_SECONDS = 3600     # từ chối video YouTube dài hơn (cùng giới hạn với website)
OUTPUT_LANGUAGE = 'en'
SOURCE_LANGUAGE = 'en'
SUMMARY_LENGTH = 'medium'

print(len(VIDEOS), 'video; kết quả ghi vào', f'{BATCH_DIR}/{RUN_NAME}.csv')

In [ ]:
import json, subprocess

sources_path = f'{BATCH_DIR}/{RUN_NAME}-sources.json'
with open(sources_path, 'w', encoding='utf-8') as f:
    json.dump([{'group': g, 'url': u} for g, u in VIDEOS], f, ensure_ascii=False, indent=1)

cmd = [
    PYTHON_BIN, '-u', '-m', 'ai_workers.batch_runner',
    '--sources', sources_path,
    '--out-csv', f'{BATCH_DIR}/{RUN_NAME}.csv',
    '--log-dir', f'{BATCH_DIR}/{RUN_NAME}-logs',      # log đầy đủ của từng video
    '--video-dir', f'{BATCH_DIR}/videos',             # <video_id>.mp4 tải sẵn, dùng khi YouTube chặn
    '--max-duration', str(MAX_VIDEO_SECONDS),
    '--output-language', OUTPUT_LANGUAGE,
    '--source-language', SOURCE_LANGUAGE,
    '--summary-length', SUMMARY_LENGTH,
    '--limit', str(LIMIT),
]
proc = subprocess.Popen(cmd, cwd='/content/mls', env=worker_env,
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
try:
    for line in proc.stdout:
        print(line, end='', flush=True)
    print('\nBatch kết thúc, mã thoát:', proc.wait())
except KeyboardInterrupt:
    proc.terminate()
    proc.wait()
    print('Đã dừng. Chạy lại cell này để tiếp tục từ video chưa xong.')

In [ ]:
import pandas as pd
from google.colab import files

csv_path = f'{BATCH_DIR}/{RUN_NAME}.csv'
xlsx_path = f'{BATCH_DIR}/{RUN_NAME}.xlsx'

df = pd.read_csv(csv_path, encoding='utf-8-sig')
df = df.drop_duplicates('url', keep='last').sort_values('index')   # lần chạy mới nhất của mỗi video
ok = df[df['status'] == 'done']
stage_cols = [c for c in df.columns if c.startswith('stage_')]

summary = pd.DataFrame({'chỉ số': [
    'số video', 'xong', 'chưa xong', 'tổng thời lượng video (giờ)', 'tổng thời gian xử lý (giờ)',
    'tổng thời gian tải (giờ)', 'hệ số realtime trung vị', 'thời gian xử lý trung bình mỗi video (giây)',
], 'giá trị': [
    len(df), len(ok), len(df) - len(ok), round(ok['video_sec'].sum() / 3600, 2),
    round(ok['processing_sec'].sum() / 3600, 2), round(df['download_sec'].sum() / 3600, 2),
    round(ok['realtime_factor'].median(), 2), round(ok['processing_sec'].mean(), 1),
]})
stage_mean = ok[stage_cols].mean().round(1).rename('trung bình (giây)').rename_axis('giai đoạn').reset_index()

with pd.ExcelWriter(xlsx_path, engine='openpyxl') as writer:
    df.to_excel(writer, sheet_name='videos', index=False)
    summary.to_excel(writer, sheet_name='tổng hợp', index=False)
    stage_mean.to_excel(writer, sheet_name='giai đoạn', index=False)

display(summary)
display(df[['index', 'group', 'video_id', 'status', 'video_sec', 'download_sec', 'processing_sec', 'realtime_factor']])
if len(df) > len(ok):
    display(df.loc[df['status'] != 'done', ['index', 'url', 'status', 'error']])
print('Đã lưu:', xlsx_path)

In [ ]:
# Tải file về máy (cả hai đều nằm sẵn trong Drive, thư mục mls-batch).
files.download(xlsx_path)
files.download(csv_path)